In [1]:
import requests
import json

url = "https://aws-prod-api-coastalpermit.gadnr.org/api/search"

params = {
    "sitePage": 1,
    "sitePageSize": 50,
    "eventPage": 1,
    "eventPageSize": 50
}

payload = {
    "keyword": "",
    "siteSearchCriteria": {},
    "eventSearchCriteria": {
        "eventType": "CMP"
    }
}

response = requests.post(url, params=params, json=payload)

print(response.status_code)
print(response.url)
print(response.text[:1000])

200
https://aws-prod-api-coastalpermit.gadnr.org/api/search?sitePage=1&sitePageSize=50&eventPage=1&eventPageSize=50
{"results":{"sites":[{"pkSiteId":11734,"commonName":"0 Estill Hammock Road","siteNumber":"12092020_044415PM","latSeconds":115236.97200,"longSeconds":-291110.40000,"dnrverified":true,"projectAddress":"0 Estill Hammock Road","altLocDesc":"","pin":"10012 01002","city":"","xrefSiteNumber":"","lotPhaseDistrict":"Portion Of The East Fork Of Estill Hammock","fkSubDivisionId":-2,"fkWaterWayId":63,"fkCountyId":28,"fkIslandId":-2,"fkWaterwayTypeId":3,"subDivisionHasRestriction":false,"subDivisionRestriction":"","waterwayHasRestriction":false,"waterwayRestriction":"","totalSiteImpacts":0.00000,"allPermitsLastExpiryDate":"2020-12-21T00:00:00","marinaLeaseLastExpiryDate":"1900-01-01T00:00:00","isAllPermitsExpired":true,"isMarinaLeaseExpired":true,"events":[{"pkEventId":27935,"fkSiteId":11734,"eventCode":"CMP20200024","description":"Empty lot with 40ft. of frontage on Chimney Creek","c

In [6]:
##Pagination test
import requests

url = "https://aws-prod-api-coastalpermit.gadnr.org/api/search"

payload = {
    "keyword": "",
    "siteSearchCriteria": {},
    "eventSearchCriteria": {
        "eventType": "CMP"
    }
}

params = {
    "sitePage": 1,
    "sitePageSize": 50,
    "eventPage": 1,
    "eventPageSize": 50
}

r = requests.post(url, params=params, json=payload)
data = r.json()

print("Top-level keys:", data.keys())
print("Results keys:", data["results"].keys())
print("Number of sites:", len(data["results"]["sites"]))

# Look at the first site
first_site = data["results"]["sites"][0]

print("\nFirst site keys:")
print(first_site.keys())

print("\nNumber of events associated with first site:",
      len(first_site.get("events", [])))

print("\nFirst event:")
print(first_site["events"][0])

Top-level keys: dict_keys(['results', 'totalSites', 'totalEvents', 'sitePage', 'sitePageSize', 'eventPage', 'eventPageSize'])
Results keys: dict_keys(['sites', 'events'])
Number of sites: 50

First site keys:
dict_keys(['pkSiteId', 'commonName', 'siteNumber', 'latSeconds', 'longSeconds', 'dnrverified', 'projectAddress', 'altLocDesc', 'pin', 'city', 'xrefSiteNumber', 'lotPhaseDistrict', 'fkSubDivisionId', 'fkWaterWayId', 'fkCountyId', 'fkIslandId', 'fkWaterwayTypeId', 'subDivisionHasRestriction', 'subDivisionRestriction', 'waterwayHasRestriction', 'waterwayRestriction', 'totalSiteImpacts', 'allPermitsLastExpiryDate', 'marinaLeaseLastExpiryDate', 'isAllPermitsExpired', 'isMarinaLeaseExpired', 'events', 'historicalTracking', 'historicalEventMemos', 'historicalDeletedSiteRecs', 'historicalCorpsPermitNums', 'historicalSiteHistory', 'isLockedForEdit', 'fkLastLockedBy', 'lastLockedAt', 'systemLockerBit', 'fkSubDivision', 'fkWaterWay', 'fkCounty', 'fkIsland', 'fkWaterwayType', 'siteStructures'

In [8]:
import os
print(os.getcwd())

C:\Users\arj26323\PhD\Jupyter\Network


In [7]:
'''##Script

import requests
import pandas as pd
import json
import time
from math import ceil

# ---------------------------------------------------------
# API setup
# ---------------------------------------------------------

url = "https://aws-prod-api-coastalpermit.gadnr.org/api/search"

payload = {
    "keyword": "",
    "siteSearchCriteria": {},
    "eventSearchCriteria": {
        "eventType": "CMP"
    }
}

page_size = 50


# ---------------------------------------------------------
# First request: determine how much data there is
# ---------------------------------------------------------

params = {
    "sitePage": 1,
    "sitePageSize": page_size,
    "eventPage": 1,
    "eventPageSize": page_size
}

r = requests.post(url, params=params, json=payload)
r.raise_for_status()

data = r.json()

total_events = data["totalEvents"]
total_sites = data["totalSites"]

print(f"Total CMPA events: {total_events}")
print(f"Total sites: {total_sites}")

n_event_pages = ceil(total_events / page_size)
n_site_pages = ceil(total_sites / page_size)

print(f"Event pages: {n_event_pages}")
print(f"Site pages: {n_site_pages}")


# ---------------------------------------------------------
# Download ALL events
# ---------------------------------------------------------

all_events = []

for page in range(1, n_event_pages + 1):

    params = {
        "sitePage": 1,
        "sitePageSize": page_size,
        "eventPage": page,
        "eventPageSize": page_size
    }

    r = requests.post(url, params=params, json=payload)
    r.raise_for_status()

    page_data = r.json()

    events = page_data["results"]["events"]

    all_events.extend(events)

    print(
        f"Event page {page}/{n_event_pages}: "
        f"{len(events)} events "
        f"(total downloaded: {len(all_events)})"
    )

    time.sleep(0.1)


# ---------------------------------------------------------
# Download ALL sites
# ---------------------------------------------------------

all_sites = []

for page in range(1, n_site_pages + 1):

    params = {
        "sitePage": page,
        "sitePageSize": page_size,
        "eventPage": 1,
        "eventPageSize": 1
    }

    r = requests.post(url, params=params, json=payload)
    r.raise_for_status()

    page_data = r.json()

    sites = page_data["results"]["sites"]

    all_sites.extend(sites)

    print(
        f"Site page {page}/{n_site_pages}: "
        f"{len(sites)} sites "
        f"(total downloaded: {len(all_sites)})"
    )

    time.sleep(0.1)


# ---------------------------------------------------------
# Convert to dataframes
# ---------------------------------------------------------

events_df = pd.json_normalize(all_events)
sites_df = pd.json_normalize(all_sites)

print("\nEvents dataframe:", events_df.shape)
print("Sites dataframe:", sites_df.shape)


# ---------------------------------------------------------
# Remove the nested event list from sites
# ---------------------------------------------------------

if "events" in sites_df.columns:
    sites_df = sites_df.drop(columns=["events"])


# ---------------------------------------------------------
# Convert nested objects/lists to JSON strings
# ---------------------------------------------------------

def stringify_nested_columns(df):

    for col in df.columns:

        if df[col].apply(
            lambda x: isinstance(x, (dict, list))
        ).any():

            df[col] = df[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False)
                if isinstance(x, (dict, list))
                else x
            )

    return df


events_df = stringify_nested_columns(events_df)
sites_df = stringify_nested_columns(sites_df)


# ---------------------------------------------------------
# Join event information to site information
# ---------------------------------------------------------

combined_df = events_df.merge(
    sites_df,
    left_on="fkSiteId",
    right_on="pkSiteId",
    how="left",
    suffixes=("_event", "_site")
)


# ---------------------------------------------------------
# Convert coordinates from arc-seconds to decimal degrees
# ---------------------------------------------------------

if "latSeconds" in combined_df.columns:
    combined_df["latitude"] = (
        combined_df["latSeconds"] / 3600
    )

if "longSeconds" in combined_df.columns:
    combined_df["longitude"] = (
        combined_df["longSeconds"] / 3600
    )


# ---------------------------------------------------------
# Remove duplicate events, just in case
# ---------------------------------------------------------

combined_df = combined_df.drop_duplicates(
    subset="pkEventId"
)


# ---------------------------------------------------------
# Save everything
# ---------------------------------------------------------

events_df.to_csv(
    "GADNR_CMPA_events.csv",
    index=False
)

sites_df.to_csv(
    "GADNR_CMPA_sites.csv",
    index=False
)

combined_df.to_csv(
    "GADNR_CMPA_events_with_sites.csv",
    index=False
)


print("\n--------------------------------")
print("DOWNLOAD COMPLETE")
print("--------------------------------")
print(f"Events: {len(events_df)}")
print(f"Sites: {len(sites_df)}")
print(f"Combined records: {len(combined_df)}")

print("\nFiles created:")
print("  GADNR_CMPA_events.csv")
print("  GADNR_CMPA_sites.csv")
print("  GADNR_CMPA_events_with_sites.csv")'''

Total CMPA events: 1207
Total sites: 686
Event pages: 25
Site pages: 14
Event page 1/25: 118 events (total downloaded: 118)
Event page 2/25: 116 events (total downloaded: 234)
Event page 3/25: 113 events (total downloaded: 347)
Event page 4/25: 113 events (total downloaded: 460)
Event page 5/25: 115 events (total downloaded: 575)
Event page 6/25: 115 events (total downloaded: 690)
Event page 7/25: 116 events (total downloaded: 806)
Event page 8/25: 115 events (total downloaded: 921)
Event page 9/25: 118 events (total downloaded: 1039)
Event page 10/25: 119 events (total downloaded: 1158)
Event page 11/25: 115 events (total downloaded: 1273)
Event page 12/25: 116 events (total downloaded: 1389)
Event page 13/25: 118 events (total downloaded: 1507)
Event page 14/25: 116 events (total downloaded: 1623)
Event page 15/25: 118 events (total downloaded: 1741)
Event page 16/25: 118 events (total downloaded: 1859)
Event page 17/25: 115 events (total downloaded: 1974)
Event page 18/25: 118 event

In [1]:
import pandas as pd
import numpy as np
dfx = pd.read_csv('C:/Users/arj26323/OneDrive - University of Georgia/Documents/Manuscripts/Network analysis/CMPA permits/GADNR_CMPA_events_with_sites.csv')

In [2]:
dfx

,pkEventId,fkSiteId,eventCode,description,corpsPermitNumber,vesselName,requestDate,conclusionDate,fkMemberIdprojectManager,fkFileLocationId,...,fkWaterWay,fkCounty,fkIsland,fkWaterwayType,siteStructures,coordinates,siteTypes,projectTypes,latitude,longitude
0,36145,660,CMP20260024,modify community dock - add float,200405400,NaN,2026-08-21T00:00:00,NaN,102,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.573370,-81.314540
1,36123,11693,CMP20260023,CMPA new community dock,202600682,NaN,2026-07-20T00:00:00,NaN,102,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.842450,-81.341230
2,35985,1283,CMP20260022,CMP 895 modification; installation of 7 additi...,202300712,NaN,2026-07-30T00:00:00,NaN,76,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],32.079108,-81.042649
3,35906,153,CMP20260021,permit modification - increase dredge depth to...,201900751,NaN,2026-07-08T00:00:00,NaN,33,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],32.078140,-81.031410
4,35893,9624,CMP20260020,CMPA for living shoreline,202600614,NaN,2026-07-08T00:00:00,NaN,99,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],30.742170,-81.588310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202,71,23,CMP19710003,484' x 20' Boat Ramp,NaN,NaN,1971-09-28T00:00:00,1971-12-01T00:00:00,-2,6,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.088400,-81.479570
1203,38,5,CMP19710002,180 Acre Secondary Waste Water Treatment Plant...,074 OYN 000294,NaN,1971-07-16T00:00:00,1971-07-16T00:00:00,-2,6,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.168750,-81.520830
1204,21,4,CMP19710001,MERGED WITH 03032016_083128AM\nFour 1-Acre Res...,TBD,NaN,1971-03-24T00:00:00,1971-03-24T00:00:00,-2,6,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.436110,-81.280550
1205,1,1,CMP19700002,Construct Secondary Treatment Wastewater Lagoon,SASKS-SH(3176),NaN,1970-11-09T00:00:00,1970-12-09T00:00:00,-2,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],32.111820,-81.118910


In [4]:
##Description will have project name terms; eventApplicantAgents will have names, company
terms = ['oyster'] ##earliest oyster bed restoration seems to be 2003 - but not a fringing reef?
# terms = ['living shoreline'] 
# terms = ['Jekyll'] 
terms = ['oyster', 'living shoreline'] ##31 matches by this combination, by coincidence?

filtered_dfx = dfx[dfx['description'].str.contains('|'.join(terms), case=False, na=False)]

xxx = filtered_dfx[['latitude', 'longitude']].copy()

filtered_dfx
##The following object ids don't exactly line up with projects that I've mapped, and I'm interested if they were moved, or if they are unrelated:
## 8, 27 (check 2 - should be Tybee LS), 30, 24, 3, 11, 12, 9, (check 10 - Belleville), 4 (Sapelo), 13, 26, 25, 5, 7

,pkEventId,fkSiteId,eventCode,description,corpsPermitNumber,vesselName,requestDate,conclusionDate,fkMemberIdprojectManager,fkFileLocationId,...,fkWaterWay,fkCounty,fkIsland,fkWaterwayType,siteStructures,coordinates,siteTypes,projectTypes,latitude,longitude
4,35893,9624,CMP20260020,CMPA for living shoreline,202600614,NaN,2026-07-08T00:00:00,NaN,99,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],30.742170,-81.588310
5,35852,2372,CMP20260019,Living Shoreline,NaN,NaN,2026-06-22T00:00:00,NaN,33,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],32.006750,-80.851817
23,35189,3665,CMP20260001,Living Shoreline,NaN,NaN,2025-11-17T00:00:00,NaN,33,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.764490,-81.144970
25,35032,13582,CMP20250048,Placement of oyster clutch materials to promot...,202300353,NaN,2025-11-18T00:00:00,2026-02-27T00:00:00,76,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.421520,-81.252910
35,34382,9294,CMP20250038,CMPA for Living Shoreline,202500592,NaN,2025-07-16T00:00:00,2025-12-02T00:00:00,99,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.209830,-81.567240
41,34229,3911,CMP20250032,excavation of northern ditch and stabilizing b...,201300402,NaN,2025-05-30T00:00:00,2025-11-14T00:00:00,76,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.150710,-81.478290
46,33998,211,CMP20250027,Living shoreline,199702940,NaN,2025-04-22T00:00:00,2025-10-15T00:00:00,76,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.229460,-81.588960
110,10374,4837,CMP20030030,Stabilize Existing Bank & Restore Oyster Beds ...,NWP13,NaN,2003-10-04T00:00:00,2003-11-20T00:00:00,-2,6,...,NaN,NaN,NaN,NaN,[],NaN,[],[],32.004190,-80.847690
123,33799,4589,CMP20250012,CMPA for a living shoreline,NaN,NaN,2025-02-26T00:00:00,NaN,99,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.640600,-81.335570
125,33727,12267,CMP20250009,CMPA for a hybrid living shoreline,202200710,NaN,2025-02-06T00:00:00,2025-10-17T00:00:00,99,-2,...,NaN,NaN,NaN,NaN,[],NaN,[],[],31.529130,-81.374460


In [41]:
# print(filtered_dfx['eventApplicantAgents'].str.split(",").str[3]) #3 is lastname, 4 first, 6 is company

In [40]:
## 8, 27 (check 2 - should be Tybee LS), 30, 24, 22, 3, 11, 12, 9, 10, 4 (Sapelo -IRIS, 100 locations), 13, 26, 25, 5, 7, 15, 14, 1

n_x = 15
n_y = n_x-1

filtered_dfx.iloc[n_y].description ##N-1

# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)

# Prevent long text inside cells from being truncated
pd.set_option('display.max_colwidth', None)
print(filtered_dfx.loc[filtered_dfx.index[n_y], ['description', 'eventApplicantAgents', 'latitude', 'longitude', 'requestDate']])

description                                                                                                                                                                                                                                                                                                                                                                                                                                                        American Oystercatcher Habitat Restoration. 202201006 = Stafford Island; 202201007 = Satilla Marsh Island. Shell Rakes
eventApplicantAgents    [{"pkEventApplicantAgentId": 52859, "fkEventId": 30783, "fkApplicantAgentTypeId": 1, "lastName": "", "firstName": "", "mi": " ", "company": "Georgia Department Of Natural Resources", "address1": "", "address2": "", "city": "", "state": "  ", "zip": "", "eventApplicantAgentContacts": []}, {"pkEventApplicantAgentId": 52860, "fkEventId": 30783, "fkApplicantAgentTypeId": 2, "lastName": "Keyes", "firstName":

In [ ]:
'''Notes on sites that didn't match up:

#4 Recent (Sapelo, oyster reef through IRIS): Under construction. 
There's a document from 2024: https://iris.uga.edu/wp-content/uploads/2026/07/Oyster-Reef-Ref2pager_V5.pdf

#8 2003 permit request, no evidence of them being built there - could they be the ones moved farther west to oyster creek?

#27 - dr alan power and marine extension. No physical location that I can find, but may be on Tybee?

#30 In Savannah. Seems they gave updates to CMPC, but hard to find spatial data anywhere else

#24: May be a news article on it, 7 years after the permit? 
https://www.savannahnow.com/story/news/2014/02/26/restoration-savannahs-mcqueens-island-trail-begins/13556315007/?gnt-cfr=1&gca-cat=p&gca-uir=true&gca-epti=z115201e115970v115201d--40--b--40--&gca-ft=140&gca-ds=sophi

#22 mistagged, likely the skidaway state park LS (but then which one is 21...)

#3 Recent grant by NOAA through CRD on Ossabaw https://coast.noaa.gov/states/stories/ossabaw-island.html

11. Recent application (2025) for LS + dock. Likely under construction.
Need to think about wording amendment to avoid just being abused for dock construction.

12. recent application (2024), likely under construction

9. recent application (2025), likely under construction

10. recent application (2025), likely under construction

13 withdrawn permit?

26 might be one of the nearby CRD-tracked reefs? honeywell, gupta, no finding it online
25 might be one of the nearby CRD-tracked reefs? marine extension, no finding it online

5 recent application (2025), likely under construction
7 recent application (2025), likely under construction

15 bird habitat

14 recent under construction, even though the permit was 2022 https://home.nps.gov/cuis/learn/news/2025-05-27-plum-orchard-dock.htm

1 recent application (2026), likely under construction


'''

In [ ]:
##RETRIEVE COMPANY NAMES FOR NETWORK ANALYSIS